In [1]:
# Import libraries required for feature engineering and machine-learning forecasting.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [2]:
df = pd.read_csv(
    "../data/hour_cleaned.csv",
    parse_dates=["datetime"]
)

df = (
    df.set_index("datetime")
      .sort_index()
)

full_index = pd.date_range(
    df.index.min(),
    df.index.max(),
    freq="h"
)

df = df.reindex(full_index)
df.index.name = "datetime"

print(f"Rows: {len(df):,}")
print(f"Missing target values: {df['cnt'].isna().sum():,}")

Rows: 17,544
Missing target values: 165


In [3]:
# Preserve the hourly sequence and create a modeling target for historical missing values.

df["was_missing"] = df["cnt"].isna()

df["target"] = df["cnt"].interpolate(
    method="time",
    limit_area="inside"
)

print(f"Remaining missing targets: {df['target'].isna().sum()}")
print(f"Originally missing: {df['was_missing'].sum()}")

Remaining missing targets: 0
Originally missing: 165


In [4]:
# Define the metrics used to compare all ML forecasting models.

def evaluate_model(actual, predicted):

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    return {
        "MAE": mae,
        "RMSE": rmse
    }

In [5]:
# Extract calendar features representing recurring demand patterns.

df["hour"] = df.index.hour
df["day_of_week"] = df.index.dayofweek
df["month"] = df.index.month
df["day_of_year"] = df.index.dayofyear
df["week_of_year"] = df.index.isocalendar().week.astype(int)

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

In [6]:
# Create lag features based on the temporal dependencies found during EDA.

lags = [
    1, 2, 3,
    6, 12,
    24, 48, 72,
    168
]

for lag in lags:
    df[f"lag_{lag}"] = df["target"].shift(lag)

In [7]:
# Calculate historical rolling statistics without using the current target.

df["rolling_mean_24"] = (
    df["target"]
    .shift(1)
    .rolling(24)
    .mean()
)

df["rolling_std_24"] = (
    df["target"]
    .shift(1)
    .rolling(24)
    .std()
)

df["rolling_mean_168"] = (
    df["target"]
    .shift(1)
    .rolling(168)
    .mean()
)

df["rolling_std_168"] = (
    df["target"]
    .shift(1)
    .rolling(168)
    .std()
)

In [8]:
# Encode hourly and weekly cycles using sine and cosine transformations.

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

df["dow_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["dow_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

In [9]:
# Define the ML feature set and remove rows without sufficient historical information.

feature_columns = [
    "temp",
    "hum",
    "windspeed",
    "weathersit",
    "workingday",
    "hour",
    "day_of_week",
    "month",
    "day_of_year",
    "week_of_year",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",
    "lag_24",
    "lag_48",
    "lag_72",
    "lag_168",
    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_168",
    "rolling_std_168"
]

ml_df = df.dropna(
    subset=feature_columns + ["target"]
).copy()

X = ml_df[feature_columns]
y = ml_df["target"]

print(f"ML rows: {len(ml_df):,}")
print(f"Features: {X.shape[1]}")

ML rows: 17,218
Features: 28


In [10]:
# Split the ML dataset chronologically into training, validation, and test sets.

n = len(ml_df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

print(f"Train:      {X_train.index.min()} → {X_train.index.max()}")
print(f"Validation: {X_val.index.min()} → {X_val.index.max()}")
print(f"Test:       {X_test.index.min()} → {X_test.index.max()}")

print(f"\nTrain rows:      {len(X_train):,}")
print(f"Validation rows: {len(X_val):,}")
print(f"Test rows:       {len(X_test):,}")

Train:      2011-01-08 00:00:00 → 2012-05-29 01:00:00
Validation: 2012-05-29 02:00:00 → 2012-09-13 16:00:00
Test:       2012-09-13 17:00:00 → 2012-12-31 23:00:00

Train rows:      12,052
Validation rows: 2,583
Test rows:       2,583


# Random Forest

In [11]:
# Train a Random Forest using the engineered temporal, weather, and lag features.

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

rf_pred = rf_model.predict(X_val)

rf_metrics = evaluate_model(
    y_val,
    rf_pred
)

print(pd.Series(rf_metrics).round(2))

MAE     40.88
RMSE    64.80
dtype: float64


Random Forest substantially outperforms all classical baselines tested so far.

The improvement indicates that combining lagged demand, rolling statistics, calendar variables, and weather information captures nonlinear demand patterns that the classical models struggled to represent.

# XGBoost

In [12]:
# Train XGBoost to model nonlinear relationships between temporal, weather, and lag features.

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)

xgb_pred = xgb_model.predict(X_val)

xgb_metrics = evaluate_model(
    y_val,
    xgb_pred
)

print(pd.Series(xgb_metrics).round(2))

MAE     42.33
RMSE    69.20
dtype: float64


XGBoost substantially outperforms the classical forecasting baselines, confirming that nonlinear relationships between lagged demand, temporal features, and weather variables are valuable.

However, the initial Random Forest configuration achieves lower MAE and RMSE than XGBoost. Hyperparameter tuning will determine whether boosting can close or reverse this gap.

# LightGBM

In [13]:
# Train LightGBM as another gradient-boosting approach for nonlinear forecasting.

lgb_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

lgb_model.fit(
    X_train,
    y_train
)

lgb_pred = lgb_model.predict(X_val)

lgb_metrics = evaluate_model(
    y_val,
    lgb_pred
)

print(pd.Series(lgb_metrics).round(2))

MAE     33.09
RMSE    51.20
dtype: float64


LightGBM achieves the best validation performance among the ML models tested so far.

Its RMSE of 51.20 substantially improves on Random Forest (64.80) and XGBoost (69.20), showing that gradient boosting with efficient tree-based learning captures the nonlinear temporal and weather relationships particularly well.